In [1]:
from azure.storage.blob import ContainerClient
account_name = "dlsaggregatedprodr9"
container_name = "saas-gold-direct-data-access"
sas_token = "sp=rl&st=2025-08-06T11:15:05Z&se=2026-07-31T19:30:05Z&spr=https&sv=2024-11-04&sr=d&sig=022LIiARMF6VCOJ1xaQGH6wv04Fwqh8quWp%2BzX07S7w%3D&sdd=2"
folder_path = "data/studio_id=10720/"
#folder_path = "data/studio_id=[your studio ID]/"


blob_service_url = f"https://{account_name}.blob.core.windows.net/"

container_client = ContainerClient(
    account_url=blob_service_url,
    container_name=container_name,
    credential=sas_token
)



In [2]:
blob_client = container_client.get_blob_client('data/studio_id=10720/export_fact_sales.csv')
blob_client_wishlist = container_client.get_blob_client('data/studio_id=10720/export_fact_visibility_wishlist.csv')
#blob_client_visibility = container_client.get_blob_client('data/studio_id=10720/fact_visibility.csv')

In [3]:
import time
import pandas as pd
from io import BytesIO


t0 = time.perf_counter()
data_rev = blob_client.download_blob().readall()
data_wl = blob_client_wishlist.download_blob().readall()
#data_vis = blob_client_visibility.download_blob().readall()

t1 = time.perf_counter()

In [4]:
df = pd.read_csv(BytesIO(data_rev))
df_wl = pd.read_csv(BytesIO(data_wl))
#df_vis = pd.read_csv(BytesIO(data_vis))

t2 = time.perf_counter()

print(f"Download: {t1 - t0:.3f}s")
print(f"CSV parse: {t2 - t1:.3f}s")
print(f"Total: {t2 - t0:.3f}s")

Download: 90.350s
CSV parse: 8.747s
Total: 99.097s


/var/folders/vh/qvl8gx_d2m370qv72srt5r1m0000gp/T/ipykernel_93621/3286014683.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_wl = pd.read_csv(BytesIO(data_wl))


In [5]:
df_wl

,date,unique_sku_id,base_sku_id,human_name,product_id,product_name,portal_platform_region_id,portal,store,country_code,non_owner_visits,non_owner_impressions,adds,deletes,purchases_activations_gifts
0,2010-01-01,1945140-store:10720,1945140,A Memoir Blue - Original Soundtrack,A Memoir Blue - Original Soundtrack:171010:10720,A Memoir Blue - Original Soundtrack,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
1,2010-01-01,1497450-store:10720,1497450,A Memoir Blue,A Memoir Blue:171010:10720,A Memoir Blue,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
2,2010-01-01,1116060-store:10720,1116060,Ashen - Nightstorm Isle,Ashen - Nightstorm Isle DLC:171010:10720,Ashen - Nightstorm Isle DLC,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
3,2010-01-01,1202450-store:10720,1202450,Ashen - Original Soundtrack,Ashen - Original Soundtrack:171010:10720,Ashen - Original Soundtrack,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
4,2010-01-01,649950-store:10720,649950,Ashen,Ashen:171010:10720,Ashen,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1340699,2025-09-23,1299460-store:10720,1299460,Wanderstop,Wanderstop:171010:10720,Wanderstop,171010,Steam,Steam,YYY,NaN,NaN,30.0,16.0,2.0
1340700,2025-09-23,1792460-store:10720,1792460,We Kill Monsters,We Kill Monsters:171010:10720,We Kill Monsters,171010,Steam,Steam,YYY,NaN,NaN,6.0,1.0,0.0
1340701,2025-09-23,501300-store:10720,501300,What Remains of Edith Finch,What Remains of Edith Finch:171010:10720,What Remains of Edith Finch,171010,Steam,Steam,YYY,NaN,NaN,83.0,9.0,1.0
1340702,2025-09-23,1497460-store:10720,1497460,Wheel World,Wheel World:171010:10720,Wheel World,171010,Steam,Steam,YYY,NaN,NaN,16.0,9.0,2.0


In [6]:
df_wl.head().columns

Index(['date', 'unique_sku_id', 'base_sku_id', 'human_name', 'product_id',
       'product_name', 'portal_platform_region_id', 'portal', 'store',
       'country_code', 'non_owner_visits', 'non_owner_impressions', 'adds',
       'deletes', 'purchases_activations_gifts'],
      dtype='object')

In [7]:
df_wl = df_wl.groupby(["date",'product_name'])[['adds','deletes','purchases_activations_gifts','non_owner_visits','non_owner_impressions']].sum().sort_values(by='date',ascending=False).reset_index()

In [8]:
df = df.groupby(["date",'product_name'])[['all_units','units_returned','units_freely_distributed','units_sold_in_retail','gross_revenue', 'gross_returned']].sum().sort_values(by='date',ascending=False).reset_index()

In [9]:
df.columns

Index(['date', 'product_name', 'all_units', 'units_returned',
       'units_freely_distributed', 'units_sold_in_retail', 'gross_revenue',
       'gross_returned'],
      dtype='object')

In [10]:
product_name = "LEGO Voyagers - Friend’s Pass"


In [11]:
df.product_name.unique()

array(['What Remains of Edith Finch', 'Lorelei and the Laser Eyes',
       'Ashen - Nightstorm Isle DLC', 'Cocoon', 'Donut County',
       'Florence', 'Gone Home', 'Gorogoa', 'Hindsight', 'I Am Dead',
       'Kentucky Route Zero: TV Edition', 'LEGO Voyagers',
       'LEGO Voyagers - Friend’s Pass', 'Ashen', 'Maquette',
       'Sayonara Wild Hearts', 'Twelve Minutes', 'The Pathless', 'Stray',
       'Solar Ash', 'Storyteller',
       'Outer Wilds - Archaeologist Edition BUNDLE', 'Outer Wilds',
       'Open Roads', 'Neon White', 'Mundaun', 'If Found...', 'Journey',
       'LEGO Voyagers - Soundtrack', 'LEGO Voyagers - Soundtrack Bundle',
       'Last Stop', 'Lushfoil Photography Sim', 'Hohokum', 'Flower',
       'Demi', 'D-topia', 'A Memoir Blue',
       'Neon White - Original Soundtrack', 'Stray - Original Soundtrack',
       'to a T', 'Wheel World',
       'What Remains of Edith Finch - Original Soundtrack', 'Wattam',
       'Wanderstop', 'Thirsty Suitors', 'The Unfinished Swan',
     

In [12]:
df.query("product_name ==@product_name")

,date,product_name,all_units,units_returned,units_freely_distributed,units_sold_in_retail,gross_revenue,gross_returned
12,2025-09-23,LEGO Voyagers - Friend’s Pass,499,0,467,0,620.81,0.0
26,2025-09-22,LEGO Voyagers - Friend’s Pass,2738,0,2454,0,6582.43,0.0
81,2025-09-21,LEGO Voyagers - Friend’s Pass,9448,0,8729,0,16605.08,0.0
143,2025-09-20,LEGO Voyagers - Friend’s Pass,10340,0,9510,0,19060.95,0.0
198,2025-09-19,LEGO Voyagers - Friend’s Pass,7364,1,6780,0,13361.27,0.0
253,2025-09-18,LEGO Voyagers - Friend’s Pass,7042,0,6566,0,10928.71,0.0
304,2025-09-17,LEGO Voyagers - Friend’s Pass,8763,0,8185,0,13422.86,0.0
368,2025-09-16,LEGO Voyagers - Friend’s Pass,10709,0,9981,0,16740.83,0.0
420,2025-09-15,LEGO Voyagers - Friend’s Pass,8682,0,7825,0,20163.79,0.0
532,2025-09-13,LEGO Voyagers - Friend’s Pass,2,0,2,0,0.00,0.0


In [13]:
data = df.merge(df_wl, on=['product_name', 'date'], how='right')

In [14]:
data.groupby("product_name")['units_returned'].sum().sort_values(ascending=False)

product_name
Stray                          258044.0
Outer Wilds                    253931.0
Journey                        126329.0
What Remains of Edith Finch    110430.0
Storyteller                     57494.0
                                 ...   
D-topia                             0.0
Demi                                0.0
Saru                                0.0
Dreamfeel Next                      0.0
Faraway                             0.0
Name: units_returned, Length: 105, dtype: float64

In [15]:
data.groupby("product_name")['all_units'].get_group("Wheel World").sum() - data.groupby("product_name")['units_freely_distributed'].get_group("Wheel World").sum() - data.groupby("product_name")['units_sold_in_retail'].get_group("Wheel World").sum() -data.groupby("product_name")['units_returned'].get_group("Wheel World").sum()

14720.0

In [16]:
data

,date,product_name,all_units,units_returned,units_freely_distributed,units_sold_in_retail,gross_revenue,gross_returned,adds,deletes,purchases_activations_gifts,non_owner_visits,non_owner_impressions
0,2025-09-23,to a T,NaN,NaN,NaN,NaN,NaN,NaN,3.0,1.0,0.0,0.0,0.0
1,2025-09-23,Flower,NaN,NaN,NaN,NaN,NaN,NaN,4.0,6.0,0.0,0.0,0.0
2,2025-09-23,Lushfoil Photography Sim,NaN,NaN,NaN,NaN,NaN,NaN,5.0,7.0,4.0,0.0,0.0
3,2025-09-23,Lorelei and the Laser Eyes,3.0,0.0,0.0,0.0,57.85,0.0,9.0,3.0,0.0,0.0,0.0
4,2025-09-23,Last Stop,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
444986,2010-01-01,Storyteller,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
444987,2010-01-01,Storyteller - Original Soundtrack,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
444988,2010-01-01,Stray,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
444989,2010-01-01,Stray - Original Soundtrack,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0


In [17]:
data['date'] = pd.to_datetime(data['date'])
#data['date_str'] = data['date'].dt.strftime("%Y-%m-%d")

In [18]:
data['units_excl_refunds'] = data['all_units'] - data['units_returned'] - data['units_freely_distributed']- data['units_sold_in_retail']
data['revenue_excl_refunds'] = data['gross_revenue'] - data['gross_returned']
data['units_excl_refunds_incl_free'] = data['all_units'] - data['units_returned']


In [19]:
grouped = data.groupby('product_name')

In [20]:
grouped.get_group("Wheel World")['units_returned'].sum()

888.0

In [21]:
rename_dict = {
'date':'Date', 
'product_name':'Product', 
'revenue_excl_refunds':'Revenue (excl. refunds)', 
'units_excl_refunds':'Units (excl. refunds)',
'units_excl_refunds_incl_free':'units_excl_refunds_incl_free',

'units_freely_distributed':'Free units', 
    'adds':'Wishlist adds', 
    'non_owner_visits':'Non-owner visits',
       'non_owner_impressions':'Non-owner impressions', 
'units_returned': 'Refunded units',
    'deletes':"wl_deletes",
    'purchases_activations_gifts':"wl_activations",
    
}

In [22]:
data.rename(rename_dict, axis=1, inplace=True)
#data = data.drop_duplicates(subset=['day', 'product'])
#data['day'] = pd.to_datetime(test['day'] )
#data = data.sort_values(by='day')

In [23]:
product_name = "LEGO Voyagers - Friend’s Pass"


In [24]:
data.query("Product ==@product_name")

,Date,Product,all_units,Refunded units,Free units,units_sold_in_retail,gross_revenue,gross_returned,Wishlist adds,wl_deletes,wl_activations,Non-owner visits,Non-owner impressions,Units (excl. refunds),Revenue (excl. refunds),units_excl_refunds_incl_free
53,2025-09-22,LEGO Voyagers - Friend’s Pass,2738.0,0.0,2454.0,0.0,6582.43,0.0,1.0,0.0,0.0,7.0,33.0,284.0,6582.43,2738.0
142,2025-09-21,LEGO Voyagers - Friend’s Pass,9448.0,0.0,8729.0,0.0,16605.08,0.0,1377.0,2.0,0.0,81.0,899.0,719.0,16605.08,9448.0
232,2025-09-20,LEGO Voyagers - Friend’s Pass,10340.0,0.0,9510.0,0.0,19060.95,0.0,2006.0,2.0,1.0,82.0,912.0,830.0,19060.95,10340.0
321,2025-09-19,LEGO Voyagers - Friend’s Pass,7364.0,1.0,6780.0,0.0,13361.27,0.0,1395.0,3.0,3.0,111.0,916.0,583.0,13361.27,7363.0
414,2025-09-18,LEGO Voyagers - Friend’s Pass,7042.0,0.0,6566.0,0.0,10928.71,0.0,1384.0,6.0,3.0,89.0,928.0,476.0,10928.71,7042.0
500,2025-09-17,LEGO Voyagers - Friend’s Pass,8763.0,0.0,8185.0,0.0,13422.86,0.0,1610.0,1.0,3.0,98.0,1076.0,578.0,13422.86,8763.0
593,2025-09-16,LEGO Voyagers - Friend’s Pass,10709.0,0.0,9981.0,0.0,16740.83,0.0,1644.0,1.0,4.0,111.0,1134.0,728.0,16740.83,10709.0
685,2025-09-15,LEGO Voyagers - Friend’s Pass,8682.0,0.0,7825.0,0.0,20163.79,0.0,1589.0,1.0,1.0,134.0,1217.0,857.0,20163.79,8682.0
774,2025-09-14,LEGO Voyagers - Friend’s Pass,NaN,NaN,NaN,NaN,NaN,NaN,2097.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
863,2025-09-13,LEGO Voyagers - Friend’s Pass,2.0,0.0,2.0,0.0,0.00,0.0,2746.0,0.0,0.0,0.0,0.0,0.0,0.00,2.0


In [25]:
data[['Date', 'Product', 'Revenue (excl. refunds)', 'Units (excl. refunds)','units_excl_refunds_incl_free',
       'Free units', 'Wishlist adds', 'Non-owner visits',
       'Non-owner impressions', 'Refunded units','wl_deletes','wl_activations']].sort_values(by='Date').query("Date=='2025-09-08' & Product ==@product_name")

/var/folders/vh/qvl8gx_d2m370qv72srt5r1m0000gp/T/ipykernel_93621/3020157460.py:3: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  'Non-owner impressions', 'Refunded units','wl_deletes','wl_activations']].sort_values(by='Date').query("Date=='2025-09-08' & Product ==@product_name")


,Date,Product,Revenue (excl. refunds),Units (excl. refunds),units_excl_refunds_incl_free,Free units,Wishlist adds,Non-owner visits,Non-owner impressions,Refunded units,wl_deletes,wl_activations
1318,2025-09-08,LEGO Voyagers - Friend’s Pass,0.0,0.0,1.0,1.0,451.0,0.0,0.0,0.0,0.0,0.0


In [26]:
export_df = data[['Date', 'Product', 'Revenue (excl. refunds)', 'Units (excl. refunds)','units_excl_refunds_incl_free',
       'Free units', 'Wishlist adds', 'Non-owner visits',
       'Non-owner impressions', 'Refunded units','wl_deletes','wl_activations']].sort_values(by='Date')

export_df.query("Date=='2025-09-18' & Product ==@product_name")

/var/folders/vh/qvl8gx_d2m370qv72srt5r1m0000gp/T/ipykernel_93621/3129069852.py:5: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  export_df.query("Date=='2025-09-18' & Product ==@product_name")


,Date,Product,Revenue (excl. refunds),Units (excl. refunds),units_excl_refunds_incl_free,Free units,Wishlist adds,Non-owner visits,Non-owner impressions,Refunded units,wl_deletes,wl_activations
414,2025-09-18,LEGO Voyagers - Friend’s Pass,10928.71,476.0,7042.0,6566.0,1384.0,89.0,928.0,0.0,6.0,3.0


In [27]:
export_df.tail(25)

,Date,Product,Revenue (excl. refunds),Units (excl. refunds),units_excl_refunds_incl_free,Free units,Wishlist adds,Non-owner visits,Non-owner impressions,Refunded units,wl_deletes,wl_activations
19,2025-09-23,A Memoir Blue,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,2.0,0.0
10,2025-09-23,Florence,21.05,13.0,13.0,0.0,23.0,0.0,0.0,0.0,4.0,1.0
20,2025-09-23,Morsels,NaN,NaN,NaN,NaN,1.0,0.0,0.0,NaN,1.0,0.0
30,2025-09-23,The Pathless,8.37,1.0,1.0,0.0,5.0,0.0,0.0,0.0,10.0,0.0
22,2025-09-23,Neon White,8.88,1.0,1.0,0.0,24.0,0.0,0.0,0.0,10.0,1.0
41,2025-09-23,Outer Wilds,241.67,18.0,18.0,0.0,160.0,0.0,0.0,0.0,39.0,15.0
40,2025-09-23,Outer Wilds - Echoes of the Eye DLC,NaN,NaN,NaN,NaN,12.0,0.0,0.0,NaN,2.0,2.0
39,2025-09-23,Sayonara Wild Hearts,7.79,1.0,1.0,0.0,3.0,0.0,0.0,0.0,5.0,0.0
38,2025-09-23,Skin Deep,NaN,NaN,NaN,NaN,6.0,0.0,0.0,NaN,3.0,1.0
37,2025-09-23,Solar Ash,29.38,2.0,2.0,0.0,5.0,0.0,0.0,0.0,2.0,0.0


In [28]:
export_df.to_csv("DB_Update/API/bulkAPI_Export.csv", index=False)

In [29]:
export_df['Product'].unique()

array(['A Memoir Blue', 'Florence', 'Ashen',
       'Ashen - Nightstorm Isle DLC', 'Ashen - Original Soundtrack',
       'BattleSage', 'Big Hops', 'Blade Runner 2033: Labyrinth',
       'Bounty Star', 'Cocoon', 'Cocoon - Soundtrack', 'Donut County',
       'Donut County - Original Soundtrack', 'Due Process',
       'Due Process - Original Soundtrack', 'Flock',
       'Florence - Original Soundtrack', 'Morsels', 'Forever Ago',
       'Flower', 'If Found...', 'Gorogoa',
       'Gorogoa - Original Soundtrack', 'Hindsight',
       'Hindsight - Original Soundtrack', 'Hohokum', 'I Am Dead',
       'Flock - Original Soundtrack', 'I Am Dead - Original Soundtrack',
       'If Found... - Original Soundtrack', 'Journey', 'Last Stop',
       'Lorelei and the Laser Eyes',
       'Lorelei and the Laser Eyes - Original Soundtrack',
       'Lushfoil Photography Sim', 'Maquette',
       'Last Stop - Original Soundtrack', 'Mixtape',
       'Stray - Original Soundtrack', 'Mundaun', 'Telling Lies',
      

In [30]:
stop

NameError: name 'stop' is not defined

In [ ]:
stop